## Comparação de Algoritmos: CPU(Quick Sort) vs GPU(Bitonic Sort)

Abaixo está a amostra do codigo em C comentado com funcoes destinadas para gpu e para cpu, as funcoes para GPU sera exclusivamente para o Bitonic Sort e as funcoes destinadas a CPU sera para o Quick sort, como trabalho proposto, o Main sera somente para a chamada das funcoes de ordencao, para medir o tempo de execucao de cada Sort e para verificaçao de ambiguidade dos resultados dos Sort's, tbm foi colocado um sistema que calcula a razao do metodo de ordenacao mais devagar para o mais rapido e exibido no final.

---

In [1]:
%%writefile sorts_cpu_gpu.cu

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// O Bitonic Sort exige que o tamanho do array seja uma potência de 2.
// 1 << 20 gera 1.048.576 elementos (excelente para estressar a CPU e GPU).
#define N (1 << 20)
#define THREADS_PER_BLOCK 512

// ==========================================
// 1. QUICKSORT NA CPU (Divisão e Conquista)
// ==========================================
void swap_cpu(int* a, int* b) {
    int t = *a;
    *a = *b;
    *b = t;
}

int partition(int arr[], int low, int high) {
    int pivot = arr[high]; // Escolhe o último elemento como pivô
    int i = (low - 1);

    for (int j = low; j <= high - 1; j++) {
        if (arr[j] < pivot) {
            i++;
            swap_cpu(&arr[i], &arr[j]);
        }
    }
    swap_cpu(&arr[i + 1], &arr[high]);
    return (i + 1);
}

void quickSortCPU(int arr[], int low, int high) {
    if (low < high) {
        int pi = partition(arr, low, high);

        // Recursão - Ideal para arquitetura serial da CPU
        quickSortCPU(arr, low, pi - 1);
        quickSortCPU(arr, pi + 1, high);
    }
}

// ==========================================
// 2. BITONIC SORT NA GPU (Rede de Ordenação)
// ==========================================

// Kernel iterativo - Executado pelas threads da GPU
__global__ void bitonic_sort_step(int *dev_values, int j, int k) {
    unsigned int i = threadIdx.x + blockDim.x * blockIdx.x;

    // Operação Bitwise (XOR) para descobrir o par de comparação
    unsigned int ixj = i ^ j;

    // Garante que apenas uma thread do par execute a troca
    if (ixj > i) {
        // Operação Bitwise (AND) para definir o sentido (crescente ou decrescente)
        if ((i & k) == 0) {
            if (dev_values[i] > dev_values[ixj]) {
                int temp = dev_values[i];
                dev_values[i] = dev_values[ixj];
                dev_values[ixj] = temp;
            }
        } else {
            if (dev_values[i] < dev_values[ixj]) {
                int temp = dev_values[i];
                dev_values[i] = dev_values[ixj];
                dev_values[ixj] = temp;
            }
        }
    }

    // Sincronização de Threads dentro do bloco
    __syncthreads();
}

void bitonicSortGPU(int *h_array, int size) {
    int *d_array;
    size_t bytes = size * sizeof(int);

    // 1. Gerenciamento de Memória Real (Alocação na VRAM)
    cudaMalloc((void**)&d_array, bytes);

    // 2. Copiar do CPU para GPU antes de ordenar
    cudaMemcpy(d_array, h_array, bytes, cudaMemcpyHostToDevice);

    int blocks = size / THREADS_PER_BLOCK;

    // 4. Loops complexos / Kernels Iterativos controlados pelo Host
    for (int k = 2; k <= size; k <<= 1) {
        for (int j = k >> 1; j > 0; j = j >> 1) {
            bitonic_sort_step<<<blocks, THREADS_PER_BLOCK>>>(d_array, j, k);

            // Força a GPU a terminar o passo atual antes do próximo loop do Host
            cudaDeviceSynchronize();
        }
    }

    // 3. Copiar de volta para a CPU após a ordenação terminar
    cudaMemcpy(h_array, d_array, bytes, cudaMemcpyDeviceToHost);

    // Libera a memória alocada na GPU
    cudaFree(d_array);
}

// ==========================================
// FUNÇÃO PRINCIPAL
// ==========================================
int main() {
    size_t bytes = N * sizeof(int);

    // Alocação de memória na CPU
    int *arr_cpu = (int*)malloc(bytes);
    int *arr_gpu = (int*)malloc(bytes);

    printf("Gerando array com %d elementos...\n", N);

    // Preenchendo ambos os arrays com exatamente os mesmos dados
    srand(time(NULL));
    for (int i = 0; i < N; i++) {
        int val = rand() % 100000;
        arr_cpu[i] = val;
        arr_gpu[i] = val;
    }

    // ---------------------------------------------------------
    // EXECUÇÃO DO QUICKSORT (CPU)
    // ---------------------------------------------------------
    printf("\nIniciando Quicksort (CPU)...\n");
    clock_t start_cpu = clock();

    quickSortCPU(arr_cpu, 0, N - 1);

    clock_t end_cpu = clock();
    double cpu_time_used = ((double) (end_cpu - start_cpu)) / CLOCKS_PER_SEC;
    printf("Tempo do Quicksort (CPU): %f segundos\n", cpu_time_used);

    // ---------------------------------------------------------
    // EXECUÇÃO DO BITONIC SORT (GPU)
    // ---------------------------------------------------------
    printf("\nIniciando Bitonic Sort (GPU)...\n");

    // Criando eventos do CUDA para precisão de tempo na GPU
    cudaEvent_t start_gpu, stop_gpu;
    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);

    cudaEventRecord(start_gpu);

    bitonicSortGPU(arr_gpu, N);

    cudaEventRecord(stop_gpu);
    cudaEventSynchronize(stop_gpu);

    float gpu_time_ms = 0;
    cudaEventElapsedTime(&gpu_time_ms, start_gpu, stop_gpu);

    double gpu_time_sec = gpu_time_ms / 1000.0; // Converte milissegundos para segundos
    printf("Tempo do Bitonic Sort (GPU): %f segundos\n", gpu_time_sec);

    // ---------------------------------------------------------
    // CÁLCULO DA RAZÃO DE DESEMPENHO (EM FLOAT)
    // ---------------------------------------------------------
    if (gpu_time_sec > 0 && cpu_time_used > 0) {
        float razao = (float)(cpu_time_used / gpu_time_sec);

        if (razao < 1.0f) {
            // Caso a CPU ganhe em cenários muito pequenos
            float razao_inversa = (float)(gpu_time_sec / cpu_time_used);
            printf("\nRazao do Bitonic Sort para Quicksort: %.2fx\n", razao_inversa);
        } else {
            // Cenário normal para grandes volumes de dados (GPU mais rápida)
            printf("\nRazao do Quicksort para Bitonic Sort: %.2fx\n", razao);
        }
    }

    // ---------------------------------------------------------
    // VERIFICAÇÃO DE INTEGRIDADE
    // ---------------------------------------------------------
    int correto = 1;
    for (int i = 0; i < N; i++) {
        if (arr_cpu[i] != arr_gpu[i]) {
            correto = 0;
            break;
        }
    }

    if (correto) {
        printf("\n[SUCESSO] Ambos os algoritmos ordenaram os dados corretamente!\n");
    } else {
        printf("\n[ERRO] Os arrays resultantes estao diferentes!\n");
    }

    // Limpeza final de memória e eventos
    free(arr_cpu);
    free(arr_gpu);
    cudaEventDestroy(start_gpu);
    cudaEventDestroy(stop_gpu);

    return 0;
}

Writing sorts_cpu_gpu.cu


In [3]:
!nvcc sorts_cpu_gpu.cu -o sorts_cpu_gpu
!./sorts_cpu_gpu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Gerando array com 1048576 elementos...

Iniciando Quicksort (CPU)...
Tempo do Quicksort (CPU): 0.202804 segundos

Iniciando Bitonic Sort (GPU)...
Tempo do Bitonic Sort (GPU): 0.007907 segundos

Razao do Quicksort para Bitonic Sort: 25.65x

[SUCESSO] Ambos os algoritmos ordenaram os dados corretamente!
